# N141 · Euler序、倍增与树上查询

**目标：** 把子树查询变成区间并按二进制跳祖先。

**先修：** N075, N124, N135, N080。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** tin/tout；子树连续；up[v][j]；LCA；跳跃上界；树链剖分扩展。

## 从问题到算法

DFS进入序把每棵子树变成连续区间[tin,tout)，注意这不是BFS顺序，也不是把每次回边都写入的完整Euler walk。倍增保存2^j步祖先，把k分解为二进制；LCA先对齐深度，再一起向上跳到最近公共祖先的下一层。

## 最小实现

**本章接口：** `euler_tour(adj, root)`、`BinaryLifting.get_kth_ancestor(v, k)`、`BinaryLifting.lca(u, v)`

In [1]:
def euler_tour(adj,root=0):
    n=len(adj)
    if not n: return [],[],[]
    if not 0<=root<n: raise IndexError('invalid root')
    tin=[-1]*n; tout=[-1]*n; order=[]; stack=[(root,-1,False)]
    while stack:
        u,parent,exiting=stack.pop()
        if exiting: tout[u]=len(order); continue
        if tin[u]!=-1: raise ValueError('tree required')
        tin[u]=len(order); order.append(u); stack.append((u,parent,True))
        for v in reversed(adj[u]):
            if v!=parent: stack.append((v,u,False))
    if len(order)!=n: raise ValueError('connected tree required')
    return tin,tout,order

class BinaryLifting:
    def __init__(self,adj,root=0):
        self.tin,self.tout,self.order=euler_tour(adj,root); self.n=len(adj); self.depth=[0]*self.n
        parent=[-1]*self.n
        for u in self.order:
            for v in adj[u]:
                if self.tin[v]>self.tin[u]: parent[v]=u; self.depth[v]=self.depth[u]+1
        self.up=[parent]
        for _ in range(1,max(1,self.n.bit_length())):
            old=self.up[-1]; self.up.append([-1 if p==-1 else old[p] for p in old])
    def get_kth_ancestor(self,v,k):
        if not 0<=v<self.n or k<0: raise ValueError('valid vertex and nonnegative k required')
        if k>self.depth[v]: return -1
        bit=0
        while k:
            if k&1: v=self.up[bit][v]
            bit+=1; k>>=1
        return v
    def lca(self,u,v):
        if not 0<=u<self.n or not 0<=v<self.n: raise IndexError('vertex outside tree')
        if self.depth[u]<self.depth[v]: u,v=v,u
        u=self.get_kth_ancestor(u,self.depth[u]-self.depth[v])
        if u==v: return u
        for j in range(len(self.up)-1,-1,-1):
            if self.up[j][u]!=self.up[j][v]: u,v=self.up[j][u],self.up[j][v]
        return self.up[0][u]

## 正确性、前提与复杂度

DFS只有完成一个子树后才进入外部节点，所以进入时刻到退出时刻之间恰为该子树全部点。up[j][v]=up[j−1][up[j−1][v]]归纳得到2^j祖先。LCA同步提升时始终保持两点在公共祖先下方且祖先不同，最后父节点必相同且最近。

**代价：** Euler序O(n)时间空间；倍增预处理O(n log n)时间空间，祖先/LCA查询O(log n)。不存在祖先用−1，k=0返回自身；k超过深度立即返回−1而不访问越界层。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

adj=[[1,2],[0,3,4],[0],[1],[1]]; tree=BinaryLifting(adj)
show_table(['顶点','tin','tout','深度','2^j祖先'],[(u,tree.tin[u],tree.tout[u],tree.depth[u],[row[u] for row in tree.up]) for u in range(len(adj))])
print('DFS进入序',tree.order,'LCA(3,4)',tree.lca(3,4))

顶点,tin,tout,深度,2^j祖先
0,0,5,0,"[-1, -1, -1]"
1,1,4,1,"[0, -1, -1]"
2,4,5,1,"[0, -1, -1]"
3,2,3,2,"[1, 0, -1]"
4,3,4,2,"[1, 0, -1]"


DFS进入序 [0, 1, 3, 4, 2] LCA(3,4) 1


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from random import Random
from collections import deque
rng=Random(141)
for n in range(1,28):
    adj=[[] for _ in range(n)]
    for v in range(1,n):
        p=rng.randrange(v); adj[p].append(v); adj[v].append(p)
    root=rng.randrange(n); lift=BinaryLifting(adj,root); parent=[-1]*n; parent[root]=root; q=deque([root])
    while q:
        u=q.popleft()
        for v in adj[u]:
            if parent[v]==-1: parent[v]=u; q.append(v)
    def chain(v):
        out=[v]
        while v!=root: v=parent[v]; out.append(v)
        return out
    for u in range(n):
        c=chain(u)
        for k in range(n+2): assert lift.get_kth_ancestor(u,k)==(c[k] if k<len(c) else -1)
        expected={v for v in range(n) if u in chain(v)}
        assert set(lift.order[lift.tin[u]:lift.tout[u]])==expected
        for v in range(n): assert lift.lca(u,v)==next(x for x in chain(v) if x in set(c))
    assert lift.get_kth_ancestor(root,1<<100)==-1
n=1500; adj=[[] for _ in range(n)]
for i in range(n-1): adj[i].append(i+1); adj[i+1].append(i)
assert BinaryLifting(adj).get_kth_ancestor(n-1,n-1)==0
print('N141: 所有本章断言通过')

N141: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 结合Fenwick处理子树更新查询。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 区分子树区间和任意路径区间。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1483. Kth Ancestor of a Tree Node（canonical）
- 236. Lowest Common Ancestor of a Binary Tree（comparison）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。